# 03 - ColBERT: Dense Retrieval via Late Interaction

This notebook walks you through:

1. Installing the extra dependencies needed for ColBERT
2. Indexing all passages (encoding + storing multi-vector embeddings in PostgreSQL)
3. Running search queries with ColBERT (MaxSim)
4. Evaluating retrieval quality against the `qrels` ground truth

> **Prerequisite:** You must have run `01_data_preparation.ipynb` first so the
> `passages`, `queries`, and `qrels` tables are populated.

---

In [ ]:
import sys
from pathlib import Path

# Resolve project root from notebook location
project_root = Path.cwd().resolve().parent
print(f"Project root: {project_root}")

# Ensure project root is importable so `src` package can be resolved from notebooks/
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Check for .env file at project root
env_path = project_root / ".env"
if env_path.exists():
    print(f"Found .env file: {env_path}")
else:
    print("No .env found at project root. Default values from config.py will be used.")
    print("You can create one manually if needed.")

## 1) Install Python dependencies

Run this once per environment.

In [ ]:
import sys
import subprocess

requirements_file = project_root / "requirements.txt"
print(f"Installing dependencies from: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)

## 2) Index passages with ColBERT

This step encodes every passage in the `passages` table and stores the resulting multi-vector embeddings (`vector[]`) in the `colbert` table.

Each passage produces one embedding per token (after punctuation filtering), with each embedding having dimension 128.

**Estimated time:** Slower than SPLADE on CPU since each passage goes through BERT individually.

You can re-run safely — already-indexed passages are skipped.

### Optional fast path - import precomputed ColBERT vectors from CSV

If you already have a full ColBERT export, you can load it directly into PostgreSQL and skip local model encoding.

This import cell tries `colbert_data.csv.gz` first, then `colbert_data.csv`.

Expected CSV format:
- `passage_id` (BIGINT)
- `embedding` (text in pgvector array format, e.g. `{"[0.1,0.2,...]","[0.3,0.4,...]"}`)

This version uses direct `COPY` into `colbert` (no staging table) to reduce disk usage.
By default it truncates and fully replaces local ColBERT vectors.

If you run this import successfully, you can skip the next `index_passages(...)` cell.

In [ ]:
import gzip
import time
from src.database.connection import get_connection

# Prefer .csv.gz when available to reduce filesystem pressure
gz_path = project_root / "src" / "colbert" / "colbert_data.csv.gz"
csv_path = project_root / "src" / "colbert" / "colbert_data.csv"
source_path = gz_path if gz_path.exists() else csv_path
print(f"Import source: {source_path}")
if not source_path.exists():
    raise FileNotFoundError(
        f"Neither {gz_path.name} nor {csv_path.name} was found in {csv_path.parent}"
    )

def format_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds / 60:.2f} min"

def log_step(step_name: str, start_time: float, global_start: float) -> None:
    step_elapsed = time.time() - start_time
    total_elapsed = time.time() - global_start
    print(
        f"[{step_name}] done in {format_duration(step_elapsed)} "
        f"(total elapsed: {format_duration(total_elapsed)})",
        flush=True,
    )

conn = get_connection()
cur = conn.cursor()

try:
    t0 = time.time()
    print("Starting ColBERT direct COPY import...", flush=True)

    # Fast and safer transaction settings for large bulk load
    step_t0 = time.time()
    cur.execute("SET LOCAL synchronous_commit = OFF")
    cur.execute("SET LOCAL work_mem = '256MB'")
    cur.execute("SET LOCAL maintenance_work_mem = '1GB'")
    log_step("Session setup", step_t0, t0)

    # Clean leftovers from previous failed attempts (if any)
    step_t0 = time.time()
    cur.execute("DROP TABLE IF EXISTS colbert_import")
    log_step("Cleanup old staging table", step_t0, t0)

    # Replace content directly in target table to avoid duplicating 64+ GB in DB files
    step_t0 = time.time()
    cur.execute("TRUNCATE TABLE colbert")
    log_step("Truncate colbert", step_t0, t0)

    step_t0 = time.time()
    print("Starting COPY directly into colbert...", flush=True)
    open_fn = gzip.open if source_path.suffix == ".gz" else open
    with open_fn(source_path, "rt", encoding="utf-8") as f:
        cur.copy_expert(
            """
            COPY colbert (passage_id, embedding)
            FROM STDIN
            WITH (FORMAT csv, HEADER true)
            """,
            f,
        )
    log_step("COPY into colbert", step_t0, t0)

    # Final validation before commit
    step_t0 = time.time()
    cur.execute("SELECT COUNT(*) FROM colbert")
    inserted_row = cur.fetchone()
    if inserted_row is None:
        raise RuntimeError("Could not read final row count from colbert")
    inserted = inserted_row[0]

    cur.execute("SELECT COUNT(*) FROM passages")
    passages_row = cur.fetchone()
    if passages_row is None:
        raise RuntimeError("Could not read row count from passages")
    passages_total = passages_row[0]

    print(f"Rows in colbert: {inserted}", flush=True)
    print(f"Rows in passages: {passages_total}", flush=True)
    if inserted != passages_total:
        print(
            "Warning: colbert row count differs from passages. "
            "This can happen if CSV does not include every passage.",
            flush=True,
        )
    log_step("Validate final row count", step_t0, t0)

    step_t0 = time.time()
    conn.commit()
    log_step("Commit transaction", step_t0, t0)

    elapsed = time.time() - t0
    print(f"Imported rows into colbert: {inserted}", flush=True)
    print(f"Done in {format_duration(elapsed)}", flush=True)

except Exception:
    conn.rollback()
    raise
finally:
    cur.close()
    conn.close()

In [ ]:
from src.colbert.indexer import index_passages

# Adjust batch size depending on your machine's RAM / VRAM
index_passages(batch_size=64)

### Verify indexing

In [2]:
import pandas as pd
import warnings
from src.database.connection import get_connection

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy.*")

conn = get_connection()
try:
    count = pd.read_sql_query("SELECT COUNT(*) AS indexed FROM colbert", conn)
    print(f"Indexed passages: {count['indexed'][0]}")

    print("\nSample ColBERT entry (first passage):")
    sample = pd.read_sql_query(
        "SELECT passage_id, array_length(embedding, 1) AS num_vectors FROM colbert ORDER BY passage_id LIMIT 5",
        conn
    )
    display(sample)
finally:
    conn.close()

2026-04-16 14:46:13,336 - INFO - Database connection established successfully.


Indexed passages: 676193

Sample ColBERT entry (first passage):


,passage_id,num_vectors
0,1,94
1,2,93
2,3,37
3,4,128
4,5,104


## 3) Search with ColBERT

ColBERT uses MaxSim scoring and now streams passage embeddings from PostgreSQL to avoid RAM spikes.

Practical tips on large indexes:
- Keep `top_k` small (for example 10)
- Start with `max_passages` (for example 50_000) and increase gradually
- Full brute-force (`max_passages=None`) is expensive and can be very slow on large collections

In [2]:
from src.colbert.search import search_bruteforce

query = "what is the speed of light"

print(f"Query: '{query}'\n")
# Start with a capped scan to keep runtime/memory stable on large indexes
results = search_bruteforce(query, top_k=10, max_passages=50_000)

for i, r in enumerate(results, 1):
    print(f"[{i}] Score={r['score']:.4f}  Passage#{r['passage_id']}")
    print(f"    {r['text'][:150]}...\n")

/home/tim/Documents/Projet_Big_Data_IR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-16 13:28:44,869 - INFO - Database connection established successfully.
2026-04-16 13:28:44,870 - INFO - Loading ColBERT encoder ('bert-base-uncased') on cpu ...
2026-04-16 13:28:45,029 - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


Query: 'what is the speed of light'



2026-04-16 13:28:45,147 - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-16 13:28:45,259 - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-16 13:28:45,389 - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-16 13:28:45,501 - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-04-16 13:28:45,685 - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-04-16 13:28:45,803 - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model

[1] Score=10.5570  Passage#11961
    General Executive Director Salary. According to a 2011 survey by Watkins Uiberall sampling exclusively from 286 nonprofits based in Tennessee and neig...

[2] Score=10.5226  Passage#7903
    DRUG CLASS AND MECHANISM: Tetracycline is a broad spectrum antibiotic, that is, it is active against many different types of bacteria. It is effective...

[3] Score=10.2403  Passage#12208
    Roger A. Pielke, Jr. has estimated that the Space Shuttle program cost about US$170 billion (2008 dollars) through early 2008; the average cost per fl...

[4] Score=10.2385  Passage#4495
    There are two types of endoplasmic reticulum: rough endoplasmic reticulum (rough ER) and smooth endoplasmic reticulum (smooth ER). Both types are pres...

[5] Score=10.0199  Passage#1750
    Greywater contains far less nitrogen than blackwater Nine-tenths of the nitrogen contained in combined wastewater derives from toilet wastes (i.e., fr...

[6] Score=9.9596  Passage#7035
    Report A

In [ ]:
# Try your own queries (keep a cap for large collections)
for q in ["how old is the earth", "who wrote hamlet", "symptoms of diabetes"]:
    results = search_bruteforce(q, top_k=3, max_passages=50_000)
    print(f"\nQuery: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r['score']:.4f} - {r['text'][:120]}...")

## Notes

- The first run downloads `bert-base-uncased` from Hugging Face (~440 MB).
- ColBERT indexing is slower than SPLADE because each passage produces multiple dense vectors instead of one sparse vector.
- Brute-force search computes MaxSim against all indexed passages. For large collections, consider approximate nearest-neighbor techniques.
- You can re-run indexing at any time; already-indexed passages are skipped automatically.